In [1]:
import string

# Символи пунктуації
punct = set(string.punctuation)

# Морфологічні правила, що використовуються для класифікації невідомих слів
noun_suffix = ["action", "age", "ance", "cy", "dom", "ee", "ence", "er", "hood", "ion", "ism", "ist", "ity", "ling", "ment", "ness", "or", "ry", "scape", "ship", "ty"]
verb_suffix = ["ate", "ify", "ise", "ize"]
adj_suffix = ["able", "ese", "ful", "i", "ian", "ible", "ic", "ish", "ive", "less", "ly", "ous"]
adv_suffix = ["ward", "wards", "wise"]


In [2]:
def assign_unk(tok):
    """
    Призначення міток для невідомих слів
    """
    # Цифри
    if any(char.isdigit() for char in tok):
        return "--unk_digit--"

    # Пунктуація
    elif any(char in punct for char in tok):
        return "--unk_punct--"

    # Великі літери
    elif any(char.isupper() for char in tok):
        return "--unk_upper--"

    # Іменники
    elif any(tok.endswith(suffix) for suffix in noun_suffix):
        return "--unk_noun--"

    # Дієслова
    elif any(tok.endswith(suffix) for suffix in verb_suffix):
        return "--unk_verb--"

    # Прикметники
    elif any(tok.endswith(suffix) for suffix in adj_suffix):
        return "--unk_adj--"

    # Прислівники
    elif any(tok.endswith(suffix) for suffix in adv_suffix):
        return "--unk_adv--"

    return "--unk--"


In [3]:
import re

text = """
The suspected shooter has been identified as an Afghan national who entered the United States
in 2021 and at some point lived in Washington state, according to multiple people familiar with
the investigation who spoke on the condition of anonymity to discuss sensitive information.
Two of those people said the suspect was Rahmanullah Lakanwal
"""

# Tokenize the text, separating words and punctuation. This regex matches either a sequence of word characters (including apostrophes)
# or any non-whitespace, non-word character (i.e., punctuation).
tokens = re.findall(r"[\w']+|[^\s\w]", text)

# Process each token and apply assign_unk
results = []
for token in tokens:
    # Only process non-empty tokens after stripping potential whitespace
    if token.strip():
        label = assign_unk(token)
        results.append(f'"{token}" - "{label}"')

# Print the results
for res in results:
    print(res)

"The" - "--unk_upper--"
"suspected" - "--unk--"
"shooter" - "--unk_noun--"
"has" - "--unk--"
"been" - "--unk--"
"identified" - "--unk--"
"as" - "--unk--"
"an" - "--unk--"
"Afghan" - "--unk_upper--"
"national" - "--unk--"
"who" - "--unk--"
"entered" - "--unk--"
"the" - "--unk--"
"United" - "--unk_upper--"
"States" - "--unk_upper--"
"in" - "--unk--"
"2021" - "--unk_digit--"
"and" - "--unk--"
"at" - "--unk--"
"some" - "--unk--"
"point" - "--unk--"
"lived" - "--unk--"
"in" - "--unk--"
"Washington" - "--unk_upper--"
"state" - "--unk_verb--"
"," - "--unk_punct--"
"according" - "--unk--"
"to" - "--unk--"
"multiple" - "--unk--"
"people" - "--unk--"
"familiar" - "--unk--"
"with" - "--unk--"
"the" - "--unk--"
"investigation" - "--unk_noun--"
"who" - "--unk--"
"spoke" - "--unk--"
"on" - "--unk--"
"the" - "--unk--"
"condition" - "--unk_noun--"
"of" - "--unk--"
"anonymity" - "--unk_noun--"
"to" - "--unk--"
"discuss" - "--unk--"
"sensitive" - "--unk_adj--"
"information" - "--unk_noun--"
"." - "--unk

In [4]:
import nltk
import nltk.corpus
from nltk.corpus import brown

nltk.download("brown")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [5]:
#nltk.download()

In [6]:
import nltk
import random
from collections import Counter

# Завантаження корпусу (приклад для Brown)
nltk.download('brown')
from nltk.corpus import brown

# Отримання тегованих речень
tagged_sents = list(brown.tagged_sents())

# Перемішування та розділення на навчальну і тестову вибірки
random.seed(42)
random.shuffle(tagged_sents)

split_point = int(len(tagged_sents) * 0.8)
train_sents = tagged_sents[:split_point]
test_sents = tagged_sents[split_point:]

# Запис у файли
def write_tagged_sentences(sentences, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for sentence in sentences:
            for word, tag in sentence:
                f.write(f"{word}\t{tag}\n")
            f.write("\n")  # Порожній рядок між реченнями

write_tagged_sentences(train_sents, "brown_training.pos")
write_tagged_sentences(test_sents, "brown_test.pos")

# Створення словника - виправлена версія
word_counter = Counter()
for sentence in train_sents:
    for word, _ in sentence:
        word_counter[word] += 1

# Збереження слів, які з'являються принаймні двічі
vocab = {word for word, count in word_counter.items() if count >= 2}

with open("brown_vocab.txt", 'w', encoding='utf-8') as f:
    for word in sorted(vocab):
        f.write(f"{word}\n")

# Створення файлу тестових слів
with open("brown_test_words.txt", 'w', encoding='utf-8') as f:
    for sentence in test_sents:
        for word, _ in sentence:
            f.write(f"{word}\n")
        f.write("\n")  # Порожній рядок між реченнями

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [7]:
# Виправлення 1: Додавання спеціальних токенів до словника
def preprocess(vocab, data_fp):
    """
    Попередня обробка даних
    """
    orig = []
    prep = []

    # Додаю спеціальний токен для кінця речення
    if "--n--" not in vocab:
        vocab["--n--"] = len(vocab)

    # Читання даних
    with open(data_fp, "r") as data_file:
        for cnt, word in enumerate(data_file):
            # Кінець речення
            if not word.split():
                orig.append(word.strip())
                word = "--n--"
                prep.append(word)
                continue

            # Обробка невідомих слів
            elif word.strip() not in vocab:
                orig.append(word.strip())
                word = assign_unk(word.strip())  # Виправлення: додано strip()
                prep.append(word)
                continue

            else:
                orig.append(word.strip())
                prep.append(word.strip())

    assert(len(orig) == len(open(data_fp, "r").readlines()))
    assert(len(prep) == len(open(data_fp, "r").readlines()))

    return orig, prep


In [8]:
import numpy as np
import pandas as pd
from collections import defaultdict
import math

# Завантаження навчального корпусу
with open("brown_training.pos", 'r') as f:
    training_corpus = f.readlines()

# Завантаження словника
with open("brown_vocab.txt", 'r') as f:
    voc_l = f.read().split('\n')

# Створення словника з індексами для слів
vocab = {}
for i, word in enumerate(sorted(voc_l)):
    vocab[word] = i

# Завантаження тестового корпусу
with open("/content/brown_test.pos", 'r') as f:
    y = f.readlines()

# Попередня обробка тестових слів
_, prep = preprocess(vocab, "/content/brown_test_words.txt")


In [9]:
prep

['According',
 'to',
 'state',
 'law',
 'a',
 'slave',
 'had',
 'to',
 'be',
 'at',
 'least',
 'thirty',
 'years',
 'old',
 'before',
 'he',
 'could',
 'be',
 'freed',
 '.',
 '--n--',
 'With',
 'tips',
 ',',
 'the',
 'girls',
 'average',
 'between',
 '$150',
 'and',
 '$200',
 'a',
 'week',
 ',',
 'depending',
 'on',
 'basic',
 'salary',
 '.',
 '--n--',
 'It',
 'was',
 'a',
 'very',
 '--unk--',
 'offer',
 '.',
 '--n--',
 'He',
 'saw',
 'the',
 'surprise',
 'in',
 'her',
 'face',
 ',',
 'and',
 'laughed',
 'as',
 'though',
 'it',
 'were',
 'the',
 '--unk--',
 'expression',
 'he',
 'had',
 'ever',
 'seen',
 '.',
 '--n--',
 'Add',
 'holes',
 'in',
 'top',
 ',',
 'forming',
 '``',
 'S',
 "''",
 'for',
 'salt',
 'and',
 '``',
 'P',
 "''",
 'for',
 'pepper',
 '.',
 '--n--',
 'He',
 'said',
 'he',
 'would',
 'not',
 'be',
 'surprised',
 'if',
 'some',
 'of',
 'the',
 'more',
 'than',
 '30',
 'members',
 'of',
 'the',
 'group',
 'are',
 'interested',
 'in',
 'running',
 'on',
 'the',
 'required

In [10]:
# Виправлення 2: Виправлена функція get_word_tag
def get_word_tag(line, vocab):
    """
    Отримання слова та його тегу з рядка корпусу.
    """
    if not line.split():
        word = "--n--"
        tag = "--s--"
        return word, tag  # Виправлення: додано повернення значення
    else:
        parts = line.split()
        if len(parts) >= 2:
            word, tag = parts[0], parts[1]
            if word not in vocab:
                # Обробка невідомих слів
                word = assign_unk(word)
            return word, tag
        else:
            # Обробка некоректних рядків
            return "--n--", "--s--"  # Виправлення: додано повернення для некоректних рядків




In [11]:
def create_dictionaries(training_corpus, vocab):
    """
    Створення словників частот.
    """
    emission_counts = defaultdict(int)
    transition_counts = defaultdict(int)
    tag_counts = defaultdict(int)

    # Початковий тег
    prev_tag = '--s--'

    for word_tag in training_corpus:
        word, tag = get_word_tag(word_tag, vocab)

        # Збільшення лічильника переходів
        transition_counts[(prev_tag, tag)] += 1

        # Збільшення лічильника емісій
        emission_counts[(tag, word)] += 1

        # Збільшення лічильника тегів
        tag_counts[tag] += 1

        # Оновлення попереднього тегу
        prev_tag = tag

    return emission_counts, transition_counts, tag_counts



In [12]:
emission_counts, transition_counts, tag_counts = create_dictionaries(training_corpus, vocab)

In [13]:
emission_counts

defaultdict(int,
            {('PPS', 'He'): 2381,
             ('VBD', 'let'): 28,
             ('PPO', 'her'): 889,
             ('VB', 'tell'): 191,
             ('PPO', 'him'): 2085,
             ('ABN', 'all'): 2078,
             ('IN', 'about'): 993,
             ('AT', 'the'): 49705,
             ('NN', 'church'): 197,
             ('.', '.'): 39045,
             ('--s--', '--n--'): 45872,
             ('NP', 'China'): 47,
             ('RB', 'never'): 513,
             ('VBD', 'tried'): 90,
             ('TO', 'to'): 11753,
             ('VB', 'integrate'): 7,
             ('NP', 'Tibet'): 7,
             ('IN', 'by'): 4000,
             ('VBG', '--unk--'): 724,
             ('NNS$', "people's"): 12,
             ('NN', 'religion'): 83,
             ('CC', 'and'): 22217,
             ('NNS', 'institutions'): 84,
             ('RB', 'Finally'): 49,
             (',', ','): 46537,
             ('PPS', 'it'): 3016,
             ('BEZ', 'is'): 8006,
             ('VBN', 'suggested'

In [14]:
def create_transition_matrix(alpha, tag_counts, transition_counts):
    """
    Створення матриці переходів A.
    """
    all_tags = sorted(tag_counts.keys())
    num_tags = len(all_tags)

    A = np.zeros((num_tags, num_tags))

    for i in range(num_tags):
        for j in range(num_tags):
            key = (all_tags[i], all_tags[j])

            count = 0
            if key in transition_counts:
                count = transition_counts[key]

            count_prev_tag = tag_counts[all_tags[i]]

            # Застосування згладжування
            A[i, j] = (count + alpha) / (count_prev_tag + alpha * num_tags)

    return A


In [15]:
A = create_transition_matrix(0.0001, tag_counts, transition_counts)


In [16]:
A

array([[3.86033166e-07, 1.08089672e-01, 3.86033166e-07, ...,
        1.15813810e-02, 3.86033166e-07, 3.86071769e-03],
       [1.43883958e-08, 1.43883958e-08, 6.61867647e-03, ...,
        1.43883958e-08, 1.43883958e-08, 4.31666264e-04],
       [5.56215192e-04, 5.56215192e-04, 6.11781095e-03, ...,
        5.56159576e-08, 5.56159576e-08, 1.72410025e-02],
       ...,
       [4.25842075e-01, 1.41942627e-05, 1.41942627e-05, ...,
        1.41942627e-05, 1.41942627e-05, 1.41942627e-05],
       [1.41942627e-05, 1.41942627e-05, 1.41942627e-05, ...,
        1.41942627e-05, 1.41942627e-05, 1.41942627e-05],
       [2.87038900e-04, 1.43512274e-08, 2.87038900e-04, ...,
        1.43526626e-04, 8.61087997e-04, 1.43512274e-08]])

In [17]:
all_tags = sorted(tag_counts.keys())
all_tags[0]

"'"

In [18]:
vocab.keys()

dict_keys(['', '!', '$.03', '$.07', '$.50', '$1', '$1,000', '$1,500', '$1.1', '$1.9', '$10', '$10,000', '$10.50', '$100', '$100,000', '$11', '$12,500', '$12.50', '$135', '$14', '$15', '$150', '$17,000', '$18', '$2', '$2,000', '$2,000,000', '$20', '$20,000', '$200', '$23,000,000', '$25', '$25,000', '$250', '$28', '$29', '$3', '$3,000', '$3,500,000', '$3.5', '$30,000', '$300', '$32,000', '$37', '$4', '$4,500,000', '$40', '$40,000', '$400', '$45', '$450', '$5', '$5,000', '$5,000,000', '$50', '$500', '$500,000', '$5000', '$538', '$60', '$600', '$65', '$7', '$700', '$75', '$8', '$80,738', '$800', '$85', '$9', '$90', '&', "'", "''", "'13", "'30s", "'48", "'49", "'52", "'55", "'58", "'61", "'em", "'im", "'n'", "'round", "'tis", '(', ')', '**ya', '**yc', '**yf', '**yl', '**zg', '**zq', ',', '-', '--', '-78-degrees', '-ism', '.', '/', '0', '0.1', '0.16', '0.2', '0.3', '0.4', '0.5', '0.6', '05', '08', '1', '1%', "1''", '1,000', '1,083,000', '1,500', '1,600', '1,700', '1-1/2', "1-1/4''", '1-a', '

In [19]:
def create_emission_matrix(alpha, tag_counts, emission_counts, vocab):
    """
    Створення матриці емісій B.
    """
    all_tags = sorted(tag_counts.keys())
    num_tags = len(tag_counts)
    num_words = len(vocab)

    B = np.zeros((num_tags, num_words))

    for i in range(num_tags):
        for j in range(num_words):
            #print(all_tags[i], vocab[j])
            key = (all_tags[i], vocab[j])

            count = 0
            if key in emission_counts:
                count = emission_counts[key]

            count_tag = tag_counts[all_tags[i]]

            # Застосування згладжування
            B[i, j] = (count + alpha) / (count_tag + alpha * num_words)

    return B


In [20]:
B = create_emission_matrix(0.0001, tag_counts, emission_counts, sorted(vocab.keys()))

In [21]:
B.shape

(451, 27049)

In [22]:
# Виправлення 3: Оновлення функцій Вітербі для роботи з невідомими словами
def initialize(states, tag_counts, A, B, corpus, vocab):
    """
    Ініціалізація алгоритму Вітербі.
    """
    num_tags = len(tag_counts)

    # Ініціалізація матриць best_probs та best_paths
    best_probs = np.zeros((num_tags, len(corpus)))
    best_paths = np.zeros((num_tags, len(corpus)), dtype=int)

    # Індекс початкового стану
    s_idx = states.index("--s--")

    # Заповнення першого стовпця best_probs
    word = corpus[0]
    word_idx = vocab.get(word, vocab.get("--unk--", 0))  # Виправлення: обробка невідомих слів

    for i in range(num_tags):
        # A та B вже в логарифмічному просторі
        best_probs[i, 0] = A[s_idx, i] + B[i, word_idx]

    return best_probs, best_paths

In [23]:
def viterbi_forward(A, B, corpus, best_probs, best_paths, vocab):
    """
    Пряме проходження алгоритму Вітербі.
    """
    num_tags = best_probs.shape[0]

    for i in range(1, len(corpus)):
        word = corpus[i]
        word_idx = vocab.get(word, vocab.get("--unk--", 0))  # Виправлення: обробка невідомих слів

        # Calculate emission scores for the current word for all tags
        emission_scores_for_word = B[:, word_idx] # Shape (num_tags,)

        # Calculate scores from previous best probabilities plus transition probabilities
        # best_probs[:, i-1] has shape (num_tags,)
        # A has shape (num_tags, num_tags)
        # By broadcasting, best_probs[:, i-1][:, None] + A results in (num_tags, num_tags) matrix
        # where element (k, j) is best_probs[k, i-1] + A[k, j]
        scores_from_prev_tags = best_probs[:, i-1][:, np.newaxis] + A # Shape (num_tags, num_tags)

        # For each current tag j, find the maximum score from any previous tag k
        # max_scores_to_current_tags will be a vector of shape (num_tags,)
        # max_scores_to_current_tags[j] = max_k (best_probs[k, i-1] + A[k, j])
        best_probs[:, i] = np.max(scores_from_prev_tags, axis=0) + emission_scores_for_word

        # To update best_paths, we need the indices of the maximums.
        # This requires argmax along the k dimension for each j
        best_paths[:, i] = np.argmax(scores_from_prev_tags, axis=0)

    return best_probs, best_paths

In [24]:
def viterbi_backward(best_probs, best_paths, corpus, states):
    """
    Зворотне проходження алгоритму Вітербі.
    """
    m = best_paths.shape[1]
    z = [None] * m
    pred = [None] * m

    # Знаходження найкращого тегу для останнього слова
    best_prob_for_last_word = float('-inf')

    for k in range(best_probs.shape[0]):
        if best_probs[k, m-1] > best_prob_for_last_word:
            best_prob_for_last_word = best_probs[k, m-1]
            z[m-1] = k

    pred[m-1] = states[z[m-1]]

    # Зворотне проходження для знаходження найкращих тегів
    for i in range(m-2, -1, -1):
        z[i] = best_paths[z[i+1], i+1]
        pred[i] = states[z[i]]

    return pred


In [25]:
def compute_accuracy(pred, y):
    """
    Обчислення точності моделі POS-тегування.
    """
    num_correct = 0
    total = 0

    for prediction, y_item in zip(pred, y):
        y_item = y_item.strip()
        word_tag_tuple = y_item.split('\t')

        if len(word_tag_tuple) != 2:
            continue

        word, tag = word_tag_tuple

        if tag == prediction:
            num_correct += 1

        total += 1

    return num_correct / total


In [26]:
# Створення словників
emission_counts, transition_counts, tag_counts = create_dictionaries(training_corpus, vocab)

# Отримання списку всіх тегів
states = sorted(tag_counts.keys())

# Параметр згладжування
alpha = 0.001

# Створення індексованого словника слів
word_to_index = {}
for i, word in enumerate(sorted(vocab.keys())):
    word_to_index[word] = i

# Додавання спеціальних токенів
for special_token in ["--n--", "--unk--", "--unk_digit--", "--unk_punct--", "--unk_upper--",
                      "--unk_noun--", "--unk_verb--", "--unk_adj--", "--unk_adv--"]:
    if special_token not in word_to_index:
        word_to_index[special_token] = len(word_to_index)

# Створення матриць переходів та емісій (у логарифмічному просторі)
A = np.log(create_transition_matrix(alpha, tag_counts, transition_counts))
B = np.log(create_emission_matrix(alpha, tag_counts, emission_counts, list(word_to_index.keys())))

In [27]:

# Ініціалізація алгоритму Вітербі
best_probs, best_paths = initialize(states, tag_counts, A, B, prep, word_to_index)


In [28]:
# Пряме проходження
best_probs, best_paths = viterbi_forward(A, B, prep, best_probs, best_paths, word_to_index)

In [29]:
# Зворотне проходження для отримання найкращої послідовності тегів
pred = viterbi_backward(best_probs, best_paths, prep, states)


In [30]:
# Обчислення точності
accuracy = compute_accuracy(pred, y)
print(f"Точність моделі: {accuracy:.4f}")


Точність моделі: 0.9459


In [31]:
import nltk
from nltk import word_tokenize
from nltk.corpus import brown

# Завантаження необхідних ресурсів
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('brown')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Приклад тексту
text = """The suspected shooter has been identified as an Afghan national who entered the United States in 2021 and at some point lived in Washington state, according to multiple people familiar with the investigation who spoke on the condition of anonymity to discuss sensitive information. Two of those people said the suspect was Rahmanullah Lakanwal"""

# Токенізація тексту
tokens = word_tokenize(text)

# POS-тегування з використанням NLTK
nltk_tags = nltk.pos_tag(tokens)

print("Результат POS-тегування NLTK:")
for word, tag in nltk_tags:
    print(f"{word}: {tag}")

import random

# Оцінка точності NLTK POS-tagger на корпусі

# Отримання тегованих речень
tagged_sents = list(brown.tagged_sents())

# Перемішування та розділення на навчальну і тестову вибірки
random.seed(42)
random.shuffle(tagged_sents)

split_point = int(len(tagged_sents) * 0.8)
train_data = tagged_sents[:split_point]
test_data = tagged_sents[split_point:]

# Навчання NLTK тегера
from nltk.tag import UnigramTagger, BigramTagger
unigram_tagger = UnigramTagger(train_data)
bigram_tagger = BigramTagger(train_data, backoff=unigram_tagger)

# Оцінка точності
nltk_accuracy = bigram_tagger.accuracy(test_data)
print(f"Точність NLTK BigramTagger: {nltk_accuracy:.4f}")

# Порівняння з нашою реалізацією
print(f"Точність нашої реалізації: {accuracy:.4f}")
print(f"Різниця: {abs(accuracy - nltk_accuracy):.4f}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Результат POS-тегування NLTK:
The: DT
suspected: JJ
shooter: NN
has: VBZ
been: VBN
identified: VBN
as: IN
an: DT
Afghan: NNP
national: JJ
who: WP
entered: VBD
the: DT
United: NNP
States: NNPS
in: IN
2021: CD
and: CC
at: IN
some: DT
point: NN
lived: VBD
in: IN
Washington: NNP
state: NN
,: ,
according: VBG
to: TO
multiple: JJ
people: NNS
familiar: JJ
with: IN
the: DT
investigation: NN
who: WP
spoke: VBD
on: IN
the: DT
condition: NN
of: IN
anonymity: NN
to: TO
discuss: VB
sensitive: JJ
information: NN
.: .
Two: CD
of: IN
those: DT
people: NNS
said: VBD
the: DT
suspect: NN
was: VBD
Rahmanullah: NNP
Lakanwal: NNP
Точність NLTK BigramTagger: 0.9169
Точність нашої реалізації: 0.9459
Різниця: 0.0290
